In [ ]:
import os
from tqdm import tqdm
import numpy as np
import cv2
import random
# import bm3d import 
from bm3d import bm3d_deblurring, BM3DProfile, gaussian_kernel, gaussian_kernel
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
glioma = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/glioma_cropped/"
meningioma = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/meningioma_cropped_enhanced/"
meningioma_path = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train_enhanced/"
pituitary_path = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train_enhanced/"
pituitary = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/pituitary_tumor_cropped_enhanced/"
Training_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/"
Test_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/test/"
Val_Classes = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/val/"
augmented_set = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/oversampled"
paths = {'glioma': glioma, 'meningioma': meningioma, 'pituitary':pituitary}
print(paths)


## Need to modify the equalization function to work on image array input

In [ ]:
def eualize_darken(image_to_crop_path):        
    
    img = cv2.imread(image_to_crop_path)    
    NOISE_FLOOR = 25

    def reduce_noise(noisy_img, noise_floor=NOISE_FLOOR):
        clean_img = np.array(noisy_img)
        clean_img[noisy_img < NOISE_FLOOR] = 0
        return clean_img

    new_img = reduce_noise(img)

    # Blurr
    new_img = cv2.bilateralFilter(new_img,3,20,20)

    img_hsv = cv2.cvtColor(new_img, cv2.COLOR_RGB2YCrCb)

    img_hsv[:, :, 0] = cv2.equalizeHist(img_hsv[:, :, 0])

    image = cv2.cvtColor(img_hsv, cv2.COLOR_YCrCb2RGB)    
    return image

# equalized_darkened_img = eualize_darken(raw_cropped_image)

In [ ]:
def gamma_correct(image, image_to_crop_path, cropped_images_folder):
        
    def gammaCorrection(src, gamma):
        invGamma = 1 / gamma

        table = [((i / 255) ** invGamma) * 255 for i in range(256)]
        table = np.array(table, np.uint8)

        return cv2.LUT(src, table)

    gamma_corrected_image = gammaCorrection(image, 1.4)
    
    # Saving
    current_image_name = image_to_crop_path.rsplit('/', 1)[1]
    Cropped_image_name = current_image_name.rsplit('.')[0] + '_enhanced' + "."+ current_image_name.split('.')[1]
    print(
        f"Writing the enhanced image {Cropped_image_name} to {cropped_images_folder}... "
    )
    Cropped_image_path = f"{cropped_images_folder}/{Cropped_image_name}"
    print("Enhanced_image_path: ", Cropped_image_path)
    saved_cropped_image = cv2.imwrite(Cropped_image_path, gamma_corrected_image)

    print("Saved!")
    
    return gamma_corrected_image
